# Retail Orders — Data Profiling & Quality Checks
**Purpose:** profile the raw retail orders file, identify data-quality issues, and produce a cleaned review file without silently discarding questionable records.

Place `retail-orders-raw.csv` in the same folder as this notebook before running it.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_FILE = Path('retail-orders-raw.csv')
df = pd.read_csv(DATA_FILE, dtype={'quantity': 'string'})
print('Rows, columns:', df.shape)
display(df.head())
display(df.dtypes.to_frame('data_type'))

Rows, columns: (12, 9)


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


,data_type
order_id,object
order_date,object
customer_segment,object
city,object
category,object
quantity,string[python]
unit_price,int64
discount_pct,float64
payment_status,object


## 1. Basic profile
Review column names, missing values, distinct counts, and exact duplicate rows.

In [ ]:
profile = pd.DataFrame({
    'data_type': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'distinct_count': df.nunique(dropna=True)
})
display(profile)
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Repeated order IDs:', int(df['order_id'].duplicated(keep=False).sum()))

,data_type,missing_count,missing_pct,distinct_count
order_id,object,0,0.00,11
order_date,object,1,8.33,10
customer_segment,object,0,0.00,4
city,object,1,8.33,7
category,object,0,0.00,3
quantity,string,0,0.00,5
unit_price,int64,0,0.00,3
discount_pct,float64,1,8.33,5
payment_status,object,0,0.00,5


Exact duplicate rows: 1
Repeated order IDs: 2


## 2. Standardize and validate fields
These checks flag records; they do not silently erase the original values.

In [ ]:
clean = df.copy()

# Normalize text categories while preserving missing values
for col in ['customer_segment', 'payment_status', 'category', 'city']:
    clean[col] = clean[col].astype('string').str.strip()
clean['customer_segment'] = clean['customer_segment'].str.title()
clean['payment_status'] = clean['payment_status'].str.title()
clean['category'] = clean['category'].str.title()

# Parse dates. dayfirst=True supports the supplied 03/01/2026 example;
# ambiguous date formats should still be confirmed with the data owner.
clean['order_date_parsed'] = pd.to_datetime(clean['order_date'], errors='coerce', dayfirst=True)
clean['quantity_num'] = pd.to_numeric(clean['quantity'], errors='coerce')
clean['discount_pct_num'] = pd.to_numeric(clean['discount_pct'], errors='coerce')
clean['unit_price_num'] = pd.to_numeric(clean['unit_price'], errors='coerce')

allowed_segments = {'Student', 'Fresher', 'Professional'}
allowed_categories = {'Learning Kit', 'Course Access', 'Mentor Session'}
allowed_statuses = {'Paid', 'Pending', 'Failed', 'Refunded'}
required_cols = ['order_id','order_date','customer_segment','city','category','quantity','unit_price','discount_pct','payment_status']

checks = pd.DataFrame(index=clean.index)
checks['missing_required_field'] = clean[required_cols].isna().any(axis=1)
checks['blank_city'] = clean['city'].isna() | clean['city'].eq('')
checks['invalid_date'] = clean['order_date_parsed'].isna() | (clean['order_date_parsed'] < pd.Timestamp('2025-01-01'))
checks['invalid_segment'] = ~clean['customer_segment'].isin(allowed_segments)
checks['invalid_category'] = ~clean['category'].isin(allowed_categories)
checks['invalid_quantity'] = clean['quantity_num'].isna() | (clean['quantity_num'] <= 0) | (clean['quantity_num'] % 1 != 0)
checks['invalid_unit_price'] = clean['unit_price_num'].isna() | (clean['unit_price_num'] < 0)
checks['invalid_discount'] = clean['discount_pct_num'].notna() & ~clean['discount_pct_num'].between(0,100)
checks['missing_discount_needs_approval'] = clean['discount_pct_num'].isna()
checks['invalid_payment_status'] = ~clean['payment_status'].isin(allowed_statuses)
checks['duplicate_order_id'] = clean['order_id'].duplicated(keep=False)
display(checks.sum().sort_values(ascending=False).to_frame('flagged_rows'))


,flagged_rows
invalid_date,5
missing_required_field,3
invalid_quantity,2
duplicate_order_id,2
blank_city,1
invalid_discount,1
missing_discount_needs_approval,1
invalid_unit_price,0
invalid_category,0
invalid_segment,0


## 3. Quality report and exception file
The output includes flags so the data owner can review exceptions. Missing discount is not automatically replaced until the business owner approves the rule.

In [ ]:
report = pd.DataFrame({
    'check': checks.columns,
    'flagged_rows': [int(checks[c].sum()) for c in checks.columns],
    'threshold': ['0 rows' for _ in checks.columns],
    'action': [
        'Review required fields; contact source owner',
        'Request missing billing city',
        'Confirm date format and correct invalid dates at source',
        'Normalize valid labels; review unknown values',
        'Normalize valid labels; review unknown values',
        'Correct quantity from source; do not guess text values',
        'Confirm price from source',
        'Correct discount or obtain approval',
        'Ask business owner whether missing means zero',
        'Normalize valid statuses; review unknown values',
        'Confirm duplicate semantics before deduplication'
    ]
})
display(report)
clean['quality_flags'] = checks.apply(lambda row: '; '.join(row.index[row].tolist()), axis=1)
clean.to_csv('retail_orders_profiled_with_flags.csv', index=False)
report.to_csv('retail_data_quality_report.csv', index=False)
print('Saved: retail_orders_profiled_with_flags.csv and retail_data_quality_report.csv')

,check,flagged_rows,threshold,action
0,missing_required_field,3,0 rows,Review required fields; contact source owner
1,blank_city,1,0 rows,Request missing billing city
2,invalid_date,5,0 rows,Confirm date format and correct invalid dates ...
3,invalid_segment,0,0 rows,Normalize valid labels; review unknown values
4,invalid_category,0,0 rows,Normalize valid labels; review unknown values
5,invalid_quantity,2,0 rows,Correct quantity from source; do not guess tex...
6,invalid_unit_price,0,0 rows,Confirm price from source
7,invalid_discount,1,0 rows,Correct discount or obtain approval
8,missing_discount_needs_approval,1,0 rows,Ask business owner whether missing means zero
9,invalid_payment_status,0,0 rows,Normalize valid statuses; review unknown values


Saved: retail_orders_profiled_with_flags.csv and retail_data_quality_report.csv


## 4. KPI examples (provisional)
These metrics are provisional until duplicate, refund, and missing-discount rules are approved.

In [ ]:
valid_quantity = clean['quantity_num'].gt(0) & (clean['quantity_num'] % 1 == 0)
valid_price = clean['unit_price_num'].ge(0)
valid_discount = clean['discount_pct_num'].between(0,100)
eligible = valid_quantity & valid_price & valid_discount & clean['order_date_parsed'].notna()
gross_sales = (clean.loc[eligible, 'quantity_num'] * clean.loc[eligible, 'unit_price_num']).sum()
net_sales = (clean.loc[eligible, 'quantity_num'] * clean.loc[eligible, 'unit_price_num'] * (1-clean.loc[eligible, 'discount_pct_num']/100)).sum()
print(f'Eligible rows for provisional sales KPIs: {int(eligible.sum())}')
print(f'Provisional gross sales (INR): {gross_sales:,.2f}')
print(f'Provisional net sales (INR): {net_sales:,.2f}')
print('Note: confirm order-level deduplication and refund policy before publishing official KPIs.')

Eligible rows for provisional sales KPIs: 4
Provisional gross sales (INR): 7,391.00
Provisional net sales (INR): 6,991.50
Note: confirm order-level deduplication and refund policy before publishing official KPIs.
